# Milestone 2 - Week 1: Data and Baseline Foundation
## (pre-labeled Hugging Face maze-curriculum dataset)

Rebuilt on **`selimaktas/maze-curriculum-dataset`** (Hugging Face, MIT license,
60k ASCII mazes with provided solutions). The labelling question is decided
exactly as demanded: **no BFS and no other search anywhere in the data
pipeline** - the label masks are replayed from the *provided* solution strings
by pure rule-following. Search appears only in Section 4, as an *evaluation*
tool for predictions, never to create labels.

This notebook is **self-contained** (runs in Colab or locally with no repo
files next to it): models are defined inline; the only dependencies are
`torch`, `numpy`, `matplotlib` and `datasets`.

Week-1 checklist:

- [x] **Dataset selection** - a documented existing dataset (Section 1)
- [x] **Data adapter** - ASCII mazes -> 3-channel tensors, provided solutions -> masks (Section 2)
- [x] **Dataset validation** - endpoints, legality of every provided solution, duplicates (Section 3)
- [x] **Evaluation** - strict legal-path metric instead of connectivity alone (Section 4)
- [x] **First baseline** - trained RecurrentCNN (Section 5, results Section 6)
- [x] **P1: simple CNN** - non-recurrent control (Section 7)
- [x] **Hyperparameters, numerical results, visual comparisons, limitations** (Sections 5-8)


In [ ]:
# ============================ 0. Setup =========================
%matplotlib inline
import json
import sys
import time
from collections import Counter, deque
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

# hint: 'datasets' (Hugging Face) is auto-installed here if missing
try:
    import datasets
except ImportError:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"],
                   check=False)
    import datasets

DEVICE = torch.device("cpu")          # Week 1 runs comfortably on CPU
SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

RESULTS = Path.cwd() / "week1_results"
RESULTS.mkdir(exist_ok=True)

print("torch", torch.__version__, "| numpy", np.__version__,
      "| datasets", datasets.__version__, "| device:", DEVICE)


## 1) Dataset selection (P0)

**Chosen dataset: `selimaktas/maze-curriculum-dataset`** (Hugging Face, MIT
license). Why this one:

- a genuinely **pre-labeled, existing** dataset - solutions are *provided*
  (move strings), so the labels are not created by our own solver;
- `60,000` mazes with a built-in curriculum: grid_size 5 (25k), 7 (20k), 9 (15k)
  navigable cells - Week 1 uses the easy 5x5 level, later weeks scale up;
- per the dataset card the mazes are generated with Wilson's algorithm, i.e.
  **perfect mazes with exactly one solution path** - labels are unambiguous;
- compact (5.4 MB, parquet) and trivially re-downloadable - reproducible.

Fields per instance:

| field | type | content |
|---|---|---|
| `prompt` | string | LLM instruction text (unused here - we work on tensors) |
| `maze` | string | ASCII maze: `#` wall, ` ` free, `S` start, `E` end |
| `solution` | string | provided solution as moves N/S/E/W |
| `grid_size` | int | navigable cells per axis; ASCII grid is `2N+1` per side |

ASCII-to-tensor geometry: one move of the solution = **2 ASCII cells**
(landing cell + the wall-gap between). The Week-1 level (grid_size 5) is an
`11 x 11` ASCII grid.

Work split: the dataset ships only `train` - we carve our own
train / val / test split with a fixed seed (Section 3).


In [ ]:
# ======================= 1. Load and document the dataset =======================
CONFIG = dict(
    dataset="selimaktas/maze-curriculum-dataset (Hugging Face, MIT license)",
    week1_subset="grid_size = 5  ->  ASCII 11x11 (the easy curriculum level)",
    n_train=4000, n_val=1000, n_test=1000, seed=0,
)
print(json.dumps(CONFIG, indent=2))

raw = datasets.load_dataset("selimaktas/maze-curriculum-dataset")["train"]
print("columns:", raw.column_names, "| rows:", len(raw))
print("curriculum levels (grid_size -> count):", dict(Counter(raw["grid_size"])))

ex = raw[0]
print("\nprompt[:52]:", ex["prompt"][:52])
print("grid_size:", ex["grid_size"], "| provided solution:", repr(ex["solution"]))
print("the ASCII maze (S = start, E = end):")
print(ex["maze"])

moves = [len(raw[i]["solution"]) for i in range(4000)]
moves = np.array(moves)
print("solution length on a 4k sample: mean %.1f | max %d moves"
      % (moves.mean(), moves.max()))
